In [ ]:
# ── Imports & environment ─────────────────────────────────────────────────
ENV["MPLBACKEND"] = "Agg"                       # headless save-to-PDF

using LinearAlgebra, Statistics, StatsBase, Random, Printf
using JLD2, CSV, DataFrames
using MCMCChains, MCMCDiagnosticTools           # get_params, Chains, ess  (ESS diagnostics)
using KernelDensity                             # rescale_kde
using GaussianProcesses, MultivariateStats, Distributions   # PCA emulator / SLR-projection generator
using StatsPlots                                # corrplot / traceplot diagnostics

using PyCall, PyPlot
include("../src/utils/scale_utils.jl"); using .ScaleUtils
include("../src/utils/gp_utils.jl");    using .GPUtils

# matplotlib needs the conda C++ runtime. `conda activate`-ing your Julia env sets
# CONDA_PREFIX automatically; otherwise set ENV["CONDA_PREFIX"] to your env's prefix.
const CONDA_PREFIX = get(() -> error("CONDA_PREFIX not set — activate your conda " *
    "Julia environment, or set ENV[\"CONDA_PREFIX\"] to its path, before running " *
    "this notebook."), ENV, "CONDA_PREFIX")
using Libdl
Libdl.dlopen(joinpath(CONDA_PREFIX, "lib", "libstdc++.so.6"), Libdl.RTLD_GLOBAL)
Libdl.dlopen(joinpath(CONDA_PREFIX, "lib", "libgcc_s.so.1"), Libdl.RTLD_GLOBAL)

matplotlib = pyimport("matplotlib"); seaborn = pyimport("seaborn")
np = pyimport("numpy"); scipy = pyimport("scipy"); pyimport("scienceplots")
PdfPages = pyimport("matplotlib.backends.backend_pdf").PdfPages
const mpl = PyPlot.matplotlib
plt.style.use(["default", "science", "no-latex"])

BLAS.set_num_threads(11); np.random.seed(99); Random.seed!(99);

In [ ]:
# ============================================================================
# CONFIG — error-model configuration registry + shared constants/helpers. Change ONE line (CONFIG);
# everything downstream (caches, projections, figure filenames) follows.
#   2×2:  error {independent, wiener} × emulator-uncertainty {codeon, codeoff},
#         on the pointwise per-year GP ice-sheet emulator.
# ============================================================================

# key => (err, code, emul, physical mcmc dir, obs serial, cache/figure suffix)
CONFIGS = Dict(
  # pointwise (per-year) GP emulator — the four calibration (error-model) configurations
  "independent_codeon_pointwise"  => (err="indep",  code=true,  emul="peryear",
        mcmc="../outputs/mcmc_indep_code", serial="seed9_indep_code",    suffix="_indep_code"),
  "wiener_codeon_pointwise"       => (err="wiener", code=true,  emul="peryear",
        mcmc="../outputs/mcmc_wiener_code",        serial="seed9_wiener_code",  suffix="_wiener_code"),
  "independent_codeoff_pointwise" => (err="indep",  code=false, emul="peryear",
        mcmc="../outputs/mcmc_indep_nocode",      serial="seed9_indep_nocode",  suffix="_indep_nocode"),
  "wiener_codeoff_pointwise"      => (err="wiener", code=false, emul="peryear",
        mcmc="../outputs/mcmc_wiener_nocode",      serial="seed9_wiener_nocode", suffix="_wiener_nocode"),
)

# ---- pick ONE ----
CONFIG = "wiener_codeoff_pointwise"

@assert haskey(CONFIGS, CONFIG) "Unknown CONFIG '$CONFIG'. Options:\n  " *
        join(sort(collect(keys(CONFIGS))), "\n  ")
cfg = CONFIGS[CONFIG]

# ---- derived paths (all follow from the registry entry) ----
hpc_data_directory      = cfg.mcmc
TAG                     = cfg.suffix
saveout_location        = "../outputs/posteriors$(TAG)"
posteriors_directory    = saveout_location
path_to_posteriors      = saveout_location
path_to_projections     = "../outputs/projections$(TAG)"
path_to_slr_projections = path_to_projections

# ---- shared constants ----
FONTSIZE     = 20.5
PARAMS       = [:vmThresh, :fricExp, :mu_scale, :stiff_scale, :gamma0, :melt_flux]
REALIZATIONS = 1:100
PARAM_BOUNDS = Dict(1=>(80000.0,180000.0), 2=>(0.1,0.333), 3=>(0.8,1.2),
                    4=>(0.8,1.2), 5=>(9620.0,471000.0), 6=>(12.0,58.0))   # physical prior support
TITLE_DICT = Dict(1=>L"\sigma_{max}", 2=>L"q", 3=>L"C_{\mu}",
                  4=>L"C_{\phi}", 5=>L"\log(\gamma_0)", 6=>L"\overline{m}")
tuple_to_idx(r, c) = 3*(r - 1) + c    # (row,col) in a 2×3 grid → parameter index 1..6
GAPPED_YEARS = collect(2030:15:2300)     # 19 analysis years (2030,2045,…,2300)

# ---- parameter scaling + shared data ----
X_raw = CSV.read("../inputs/data/Amery_Input_Parameters_Filtered.csv", DataFrame)
_, _, X_mins, X_maxs = ScaleUtils.get_scaled_matrix(X_raw, filter(c -> c != :Column1, Symbol.(names(X_raw))))
unscale_params(θ) = [θ[i] * (X_maxs[p] - X_mins[p]) + X_mins[p] for (i, p) in enumerate(PARAMS)]  # scaled → physical

θ_posterior_2015 = np.load("../inputs/params/present_day_2015_posterior_samples.npy")   # this run's seed-9 present-day (2015) posterior (full 100k)

# ---- SLR projection helpers (shared by SLR Learning / Sensitivity / Posterior-Convergence) ----
all_years_no_gap = collect(2016:2300)                         # 285 annual projection steps (2016–2300)
yrs_dict = Dict(yr => idx for (idx, yr) in enumerate(all_years_no_gap))
# mean, 5th, 95th percentile of a projection matrix's `row` (one prediction year)
calculate_quantiles_mean(sample, row) = (s = sample[row, :]; (mean(s), quantile(s, 0.05), quantile(s, 0.95)))

# canonical 2015 SLR baseline (285×5000, mm) — same data the .npy held; used by SLR Learning + Posterior-Convergence
slr_2015_baseline = JLD2.load("../inputs/params/2015_SLR_Projections_$(cfg.code ? "codeon" : "codeoff").jld2", "present2015_post_mm_ssp5")

# per-realization "true" (generating) parameters, physical units — configuration-independent, cached once
# θ_mat[r] = the true (generating) parameters for realization r — read straight from the single
# source of truth (the 100 physical draws in inputs/params); no emulator_pred cache.
θ_draws = convert(Matrix{Float64}, np.load("../inputs/params/present_day_2015_posterior_100_draws.npy"))
θ_mat = zeros(length(REALIZATIONS), 6)
for (i, r) in enumerate(REALIZATIONS)
    θ_mat[i, :] = θ_draws[r, :]
end

println("CONFIG = $(CONFIG)")
println("  err=$(cfg.err)  emulator-uncertainty=$(cfg.code)  emulator=$(cfg.emul)")
println("  mcmc=$(hpc_data_directory)   posteriors=$(posteriors_directory)   fig-suffix='$(TAG)'")

In [ ]:
# Build one realization's posterior cache: scan its MCMC task-dirs, pull each year's chain into a
# Dict{year => iters×6 matrix}, save to <saveout>/R_<r>_Posterior_Dict.jld2. Returns #years found.
#   Tree read:  <HPC_Results_Folder>/output_R<r>_job<jid>_task<t>/chains_dir/<year>_chain.jld2
function load_data(realization, HPC_Results_Folder, saveout_directory)
    post_data = Dict{Int64, Matrix{Float64}}()

    for dir in readdir(HPC_Results_Folder)
        occursin("_R$(realization)_", dir) || continue
        chain_path = joinpath(HPC_Results_Folder, dir, "chains_dir")
        isdir(chain_path) || continue
        for chain_file in filter(f -> endswith(f, "_chain.jld2"), readdir(chain_path))
            year  = parse(Int, first(chain_file, 4))
            cm    = get_params(JLD2.load(joinpath(chain_path, chain_file), "chain"))
            post_data[year] = reduce(hcat, (getproperty(cm, p) for p in PARAMS))   # iters × 6
        end
    end

    mkpath(saveout_directory)
    @save "$(saveout_directory)/R_$(realization)_Posterior_Dict.jld2" post_data
    return length(post_data)
end

In [ ]:
# Build (once per configuration) each realization's posterior cache from its MCMC chains.
# Idempotent + resumable: skips realizations already cached with all 19 years, (re)builds the rest.
# Set FORCE_REBUILD = true to rebuild every realization from scratch.
FORCE_REBUILD  = false
EXPECTED_YEARS = length(GAPPED_YEARS)

mkpath(saveout_location)
built = Int[]; skipped = Int[]; incomplete = Int[]
for r in REALIZATIONS
    cache = "$(saveout_location)/R_$(r)_Posterior_Dict.jld2"
    if !FORCE_REBUILD && isfile(cache) && length(JLD2.load(cache, "post_data")) == EXPECTED_YEARS
        push!(skipped, r); continue
    end
    n = load_data(r, hpc_data_directory, saveout_location)
    push!(built, r)
    n == EXPECTED_YEARS || push!(incomplete, r)
end
println("Posterior caches → $(saveout_location)")
println("  built $(length(built)),  skipped $(length(skipped)) already-complete.")
isempty(incomplete) || @warn "Still <$(EXPECTED_YEARS) analysis years (MCMC not finished for these):" incomplete

In [ ]:
# ── DIAGNOSTIC (opt-in): MCMC trace + corner (pairwise) plots ─────────────────
# Convergence check for a specific realization/years — NOT part of the figure pipeline.
# Extracted to notebooks/diagnostics/trace_corner_plots.jl. To use, uncomment and edit:
#
# include("diagnostics/trace_corner_plots.jl")
# trace_corner(hpc_data_directory; realization=67, years=[2060,2075,2090,2105], tag=TAG)
#
# → writes ../outputs/plots/Trace_Plots/R67_<yr>_trace<TAG>.pdf and .../Pairwise/R67_<yr>_corner<TAG>.pdf

################################################################################
################################################################################
################################################################################

# Kernel Density Plots

################################################################################
################################################################################
################################################################################

In [ ]:
# Prior curves for the KDE overlay (cell 12). Distributions match the calibration priors
# (recalibration.jl); grids span each parameter's support, nudged off the exact edges.
sample_size = 5000
_grid(lo, hi) = collect(np.linspace(lo + 1e-5*(hi-lo), hi - 1e-5*(hi-lo), sample_size))

vmThresh_vals    = _grid(PARAM_BOUNDS[1]...)
fricExp_vals     = _grid(PARAM_BOUNDS[2]...)
mu_scale_vals    = _grid(PARAM_BOUNDS[3]...)
stiff_scale_vals = _grid(PARAM_BOUNDS[4]...)
gamma0_vals      = _grid(PARAM_BOUNDS[5]...)
melt_flux_vals   = _grid(PARAM_BOUNDS[6]...)

vmThresh_prior    = collect(scipy.stats.truncnorm.pdf(vmThresh_vals,    a=-3, b=3, loc=130000, scale=50000/3))
fricExp_prior     = collect(scipy.stats.trapezoid.pdf(fricExp_vals, c=0.05/0.233, d=0.18/0.233, loc=0.1, scale=0.233))
mu_scale_prior    = collect(scipy.stats.truncnorm.pdf(mu_scale_vals,    a=-2, b=2, loc=1.0, scale=0.1))
stiff_scale_prior = collect(scipy.stats.truncnorm.pdf(stiff_scale_vals, a=-2, b=2, loc=1.0, scale=0.1))
gamma0_prior      = collect(scipy.stats.truncnorm(a=np.log(9620)-10, b=np.log(471000)-10, loc=10, scale=1).pdf(np.log(gamma0_vals)) ./ gamma0_vals)
melt_flux_prior   = collect(scipy.stats.truncnorm.pdf(melt_flux_vals,  a=-2, b=2, loc=35, scale=11.5))

θ_prior = [vmThresh_prior;; fricExp_prior;; mu_scale_prior;; stiff_scale_prior;; gamma0_prior;; melt_flux_prior];

In [ ]:
# KDE for one parameter's posterior, renormalised to integrate to 1 over the plotting window [a,b].
# Returns (x, y): the sorted samples within [a,b] and their rescaled density (for ax.plot).
function rescale_kde(data, a, b)
    data_sorted = sort(data)
    kd = KernelDensity.kde(data_sorted)

    # CDF from the KDE at the samples → probability mass in [a,b]
    cdf = cumsum(pdf(kd, data_sorted)); cdf ./= cdf[end]
    idx_b = min(searchsortedfirst(data_sorted, b), length(data_sorted))
    prob_a_to_b = cdf[idx_b] - cdf[searchsortedfirst(data_sorted, a)]

    # keep samples in [a,b] and rescale their density so the curve integrates to 1 there
    filtered = data_sorted[a .<= data_sorted .<= b]
    return filtered, pdf(kd, filtered) ./ prob_a_to_b
end

In [ ]:
# KDE plotting helpers for the parametric-posterior figure. Plot window per parameter is
# PARAM_BOUNDS, except γ₀ (idx 5) which is drawn on a log axis.
kde_window(i) = i == 5 ? (log(PARAM_BOUNDS[5][1]), log(PARAM_BOUNDS[5][2])) : PARAM_BOUNDS[i]

# previous paper's present-day (2015) posterior for one parameter (red curve)
function plot_previous(previous_post, ax, var_idx)
    vals = previous_post[:, var_idx]
    a, b = kde_window(var_idx)
    x, y = rescale_kde(var_idx == 5 ? np.log(vals) : vals, a, b)
    ax.plot(x, y, color="red", label="Present-day (2015) posterior", lw=1.75)
end

# calibrated posterior for one parameter, one curve per analysis year (colored by year via `ynorm`)
function plot_one_var_posterior(post_dict, years, ax, var_idx, cmaplist, ynorm)
    a, b = kde_window(var_idx)
    for yr in years
        vals = post_dict[yr][:, var_idx]
        x, y = rescale_kde(var_idx == 5 ? np.log(vals) : vals, a, b)
        ax.plot(x, y, lw=1.8, color=cmaplist(ynorm(yr)))
    end
end

In [ ]:
# One KDE page for a realization: 2×3 posterior densities (per year), the 2015 posterior (red),
# the prior (green dashed) and the generating θ (black dashdot); analysis-year colorbar; save to `pdf`.
# Each panel also carries an INSET — the same curves with the y-axis zoomed (top = `inset_yfac` × the
# shortest posterior peak) — so the broad/low densities that the tall late-year spikes flatten stay
# readable. Each inset is auto-placed on the side away from that panel's tallest peak, keeps the
# parent's x ticks, inherits the subplot aspect ratio, and is tied to the indicator box by a
# funnel meeting the box's TOP corners (as in Fig. 8 of the manuscript).
function plot_all_Rs_KDE_pdf(real, post_dict, years, cmaplist, ynorm, prev, pdf, theta;
                             show_title=true, cbar_years=years,
                             cbar_ticks=collect(2030:45:2300),          # colorbar labelled every 45 yr
                             inset=true, inset_side=0.42, inset_y0=0.50, inset_pad=0.035,
                             inset_yfac=3.00,
                             # per-panel overrides, keyed by parameter index:
                             # 1=(a) σmax  2=(b) q  3=(c) Cμ  4=(d) Cφ  5=(e) log γ0  6=(f) m̄
                             inset_dx=Dict(2=>0.06, 6=>0.06),                    # horizontal nudge
                             inset_size=Dict{Int,Tuple{Float64,Float64}}(),      # per-panel (w,h); empty ⇒ all use inset_side
                             inset_align=Dict(1=>(:left, 80000.0),               # snap an edge to a
                                              3=>(:left, 0.9), 4=>(:right, 1.2), #   data grid line
                                              5=>(:left, 11.0)),
                             inset_xticks=Dict(2=>[0.10,0.15,0.22,0.28,0.33],
                                               5=>[9.2,10.0,11.0,12.0,13.0]),
                             inset_yticks=Dict(1=>[0.3e-4, 0.6e-4, 0.9e-4]))
    fig, ax = PyPlot.subplots(nrows=2, ncols=3, figsize=(18, 11), dpi=300,
        gridspec_kw=Dict("height_ratios"=>[1,1], "width_ratios"=>[1,1,1], "wspace"=>0.175, "hspace"=>0.25))

    show_title && fig.suptitle("Realization $real"; fontsize=30, fontweight="bold", y=0.96)

    tick_dict = Dict(1=>[80000,100000,120000,140000,160000,180000], 2=>[0.1,0.15,0.215,0.28,0.333],
                     3=>[0.8,0.9,1,1.1,1.2], 4=>[0.8,0.9,1,1.1,1.2], 5=>[9.17,10,11,12,13.06], 6=>[12,20,30,40,50,58])
    letters = ["a","b","c","d","e","f"]

    for i in 1:2, j in 1:3
        var_idx = tuple_to_idx(i, j); axij = ax[i, j]
        plot_one_var_posterior(post_dict, years, axij, var_idx, cmaplist, ynorm)   # per-year posteriors (colored)
        plot_previous(prev, axij, var_idx)                                  # 2015 posterior (red)

        axij.set_title(TITLE_DICT[var_idx], fontsize=1.2*FONTSIZE, pad=10)
        axij.ticklabel_format(style="sci", scilimits=(-2,2), useMathText=true)
        axij.tick_params(axis="both", which="major", labelsize=FONTSIZE*0.8)
        axij.xaxis.offsetText.set_fontsize(FONTSIZE*0.8); axij.yaxis.offsetText.set_fontsize(FONTSIZE*0.8)
        axij.set_xticks(tick_dict[var_idx]); axij.locator_params(tight=true, nbins=6)
        axij.grid(true, alpha=0.5, zorder=1)
        axij.text(j == 1 ? -0.13 : -0.1, 1.095, "($(letters[var_idx]))", ha="left", va="top",
                  transform=axij.transAxes, fontsize=FONTSIZE)
    end

    # priors (green dashed); γ0 on a log axis with density × jacobian
    ax[1,1].plot(vmThresh_vals,    θ_prior[:,1], lw=1.5, ls="dashed", color="green", label="Prior")
    ax[1,2].plot(fricExp_vals,     θ_prior[:,2], lw=1.5, ls="dashed", color="green")
    ax[1,3].plot(mu_scale_vals,    θ_prior[:,3], lw=1.5, ls="dashed", color="green")
    ax[2,1].plot(stiff_scale_vals, θ_prior[:,4], lw=1.5, ls="dashed", color="green")
    ax[2,2].plot(np.log(gamma0_vals), θ_prior[:,5] .* gamma0_vals, lw=1.5, ls="dashed", color="green")
    ax[2,3].plot(melt_flux_vals,   θ_prior[:,6], lw=1.5, ls="dashed", color="green")

    # generating θ (black dashdot); γ0 in log
    ax[1,1].axvline(theta[1], lw=1.75, color="black", ls="dashdot", label="Sample from present-day posterior")
    ax[1,2].axvline(theta[2], lw=1.75, color="black", ls="dashdot")
    ax[1,3].axvline(theta[3], lw=1.75, color="black", ls="dashdot")
    ax[2,1].axvline(theta[4], lw=1.75, color="black", ls="dashdot")
    ax[2,2].axvline(np.log(theta[5]), lw=1.75, color="black", ls="dashdot")
    ax[2,3].axvline(theta[6], lw=1.75, color="black", ls="dashdot")

    # ── per-panel inset (Fig. 8 style): same curves with the y-axis zoomed; an indicator box spanning
    # the full x-range; a funnel whose two lines meet the box's TOP corners. The inset keeps the parent's
    # x ticks (unless overridden), drops the 0 y-tick, and an equal w/h fraction gives it the subplot aspect.
    if inset
        fig.canvas.draw()                      # transforms must be current for the edge alignment below
        _axx(axij, xd) = axij.transAxes.inverted().transform(axij.transData.transform((xd, 0.0)))[1]
        for i in 1:2, j in 1:3
            var_idx = tuple_to_idx(i, j); axij = ax[i, j]; a, b = kde_window(var_idx)
            # density curves only — axvline's Line2D carries just 2 points, so it is skipped here
            curves = [(l.get_xdata(), l.get_ydata(), l.get_color(), l.get_linestyle(), l.get_linewidth())
                      for l in axij.get_lines() if length(l.get_ydata()) > 2]
            isempty(curves) && continue
            peaks = [maximum(c[2]) for c in curves if c[3] != "green"]       # posteriors (exclude the prior)
            isempty(peaks) && (peaks = [maximum(c[2]) for c in curves])
            ytop = inset_yfac * minimum(peaks)                              # reveal the shortest posterior
            w, h = get(inset_size, var_idx, (inset_side, inset_side))
            # default: the side AWAY from the tallest peak, so the inset never covers the spike
            gi = argmax([maximum(c[2]) for c in curves])
            xd = collect(Float64, curves[gi][1]); yd = collect(Float64, curves[gi][2])
            x0 = (xd[argmax(yd)] - a) / (b - a) > 0.5 ? inset_pad : 1 - inset_pad - w
            x0 += get(inset_dx, var_idx, 0.0)
            if haskey(inset_align, var_idx)                                 # snap an edge to a data value
                side, xv = inset_align[var_idx]; xa = _axx(axij, xv)
                x0 = side == :right ? xa - w : xa
            end
            x0 = clamp(x0, 0.0, 1.0 - w)
            axins = axij.inset_axes([x0, inset_y0, w, h]; xlim=(a, b), ylim=(0.0, ytop))
            for (x, y, col, ls, lw) in curves
                axins.plot(x, y; color=col, linestyle=ls, linewidth=lw)
            end
            axins.axvline(var_idx == 5 ? np.log(theta[5]) : theta[var_idx], lw=1.4, color="black", ls="dashdot")
            axins.ticklabel_format(style="sci", scilimits=(-2,2), useMathText=true)
            axins.set_xticks(get(inset_xticks, var_idx, tick_dict[var_idx]))
            if haskey(inset_yticks, var_idx)
                axins.set_yticks(inset_yticks[var_idx])
            else                                                            # drop the 0 y-tick
                yt = [t for t in axins.get_yticks() if t > 1e-12 && t <= ytop*1.001]
                !isempty(yt) && axins.set_yticks(yt)
            end
            axins.tick_params(axis="both", which="major", labelsize=FONTSIZE*0.55)
            axins.xaxis.offsetText.set_fontsize(FONTSIZE*0.55); axins.yaxis.offsetText.set_fontsize(FONTSIZE*0.55)
            axins.grid(true, alpha=0.4, zorder=1)
            # indicator box, then our own two connectors to its TOP corners. (matplotlib's
            # indicate_inset_zoom links *matching* corners, which splits them across y=0 and y=ytop.)
            ind = axij.indicate_inset(bounds=[a, 0.0, b - a, ytop], inset_ax=axins,
                                      edgecolor="0.5", facecolor="none", alpha=1.0, lw=1)
            for cc in ind.connectors; cc.set_visible(false); end
            for (xyA, xB) in (((0,0), a), ((1,0), b))
                cp = mpl.patches.ConnectionPatch(xyA=xyA, coordsA=axins.transAxes,
                                                 xyB=(xB, ytop), coordsB=axij.transData,
                                                 arrowstyle="-", color="0.5", lw=1, zorder=3)
                axij.add_patch(cp)
            end
        end
    end

    # analysis-year colorbar — always the full 19-year scale (`cbar_years`), even when
    # only a sparse subset of curves (`years`) is drawn; colors come from the shared `ynorm`.
    # Labels are thinned to `cbar_ticks` (every 45 yr) so they stay legible.
    sm = PyPlot.matplotlib.cm.ScalarMappable(norm=ynorm, cmap=cmaplist); sm.set_array(cbar_years)
    cbar = plt.colorbar(sm, cax=fig.add_axes([0.928, 0.25, 0.02, 0.5]), pad=0.1)
    cbar.set_ticks(cbar_ticks); cbar.set_ticklabels(["$(y)" for y in cbar_ticks])
    cbar.ax.set_title("Analysis \nyear", fontsize=FONTSIZE*0.8, pad=20)
    cbar.ax.tick_params(axis="both", which="major", labelsize=FONTSIZE*0.8)

    handles, labels = ax[1,1].get_legend_handles_labels()
    fig.legend(handles[1:3], labels[1:3], loc="lower center", ncol=6, fontsize=FONTSIZE,
               bbox_to_anchor=(0.512, 0.0), bbox_transform=PyPlot.gcf().transFigure, frameon="True", framealpha=1)

    pdf.savefig(fig); PyPlot.close(fig)
end

In [ ]:
# KDE pages. Toggle the realization set:
# REALIZATIONS_TO_PLOT = REALIZATIONS  # all 100, all 19 curves, titled  → KDE_plots_all_Realizations
REALIZATIONS_TO_PLOT = [58]          # one realization, sparse 7 curves, untitled → KDE_plots_Realization_test

is_test    = length(REALIZATIONS_TO_PLOT) == 1
plot_years = is_test ? collect(range(2030, step=45, length=7)) : GAPPED_YEARS   # sparse in test for legibility
figure_save_path = "../outputs/plots/KDE_Plots/" *
    (is_test ? "KDE_plots_Realization_test$(TAG).pdf" : "KDE_plots_all_Realizations$(TAG).pdf")
mkpath(dirname(figure_save_path))

# color scale ALWAYS spans the full 19 analysis years, regardless of how many curves are drawn
year_norm = PyPlot.matplotlib.colors.Normalize(vmin=minimum(GAPPED_YEARS), vmax=maximum(GAPPED_YEARS))
cmap      = plt.get_cmap("cool")

pdf_pages = PdfPages(figure_save_path)
try
    for r in REALIZATIONS_TO_PLOT
        post_dict = JLD2.load("$(posteriors_directory)/R_$(r)_Posterior_Dict.jld2", "post_data")
        θ_un = θ_draws[r, :]
        plot_all_Rs_KDE_pdf(r, post_dict, plot_years, cmap, year_norm, θ_posterior_2015, pdf_pages, θ_un;
                            show_title = !is_test, cbar_years = GAPPED_YEARS)
        println("Done realization $r")
    end
finally
    pdf_pages.close()
end
println("Wrote $figure_save_path")

In [ ]:
# # ── DIAGNOSTIC (opt-in): MCMC effective sample size (ESS) ─────────────────────
# # Per-parameter ESS by analysis year from the posterior caches — NOT part of the
# # figure pipeline. Extracted to notebooks/diagnostics/ess_report.jl. To use, uncomment:

# include("diagnostics/ess_report.jl")
# ess_scan(posteriors_directory; realizations=REALIZATIONS, years=GAPPED_YEARS)   # flag any param ESS < 1000 across the batch
# ess_report(posteriors_directory; realization=4, years=GAPPED_YEARS)             # full ESS/mean/sd table for one realization

# # → prints to stdout; ess_scan returns the flagged (realization, year) pairs.

################################################################################
################################################################################
################################################################################

# Parametric Learning Plots

################################################################################
################################################################################
################################################################################

In [ ]:
# 90%-CI width (q_hi − q_lo) per parameter for a posterior sample matrix `mat` (iters × 6).
# γ₀ (column 5) is measured in log-space, matching the KDE/learning convention. Returns a 6-vector.
function ci_width(mat, hi=0.95, lo=0.05)
    [ let col = (k == 5 ? log.(mat[:, k]) : mat[:, k])
          quantile(col, hi) - quantile(col, lo)
      end for k in 1:size(mat, 2) ]
end

In [ ]:
# 90%-CI width of the present-day (2015) posterior for each parameter — the learning baseline.
yr_2015_widths = ci_width(θ_posterior_2015)

In [ ]:
# Per-realization 90%-CI widths → total_widths[realization, year-slot, parameter].
# Slot 1 = 2015 baseline; slots 2..20 = GAPPED_YEARS. Loading 100 posterior caches is slow, so
# cache the (tiny) result — but ONLY when complete; while the batch runs, missing (r,year) cells
# stay NaN and the cache is skipped so it refreshes on re-run. FORCE_WIDTHS=true forces recompute.
FORCE_WIDTHS = false
widths_cache = "$(posteriors_directory)/total_widths.jld2"
years_slots  = [2015; GAPPED_YEARS]                              # 20 slots
want_shape   = (length(REALIZATIONS), length(years_slots), 6)

cached = (!FORCE_WIDTHS && isfile(widths_cache)) ? JLD2.load(widths_cache, "total_widths") : nothing
if cached !== nothing && size(cached) == want_shape
    total_widths = cached
    println("Loaded cached total_widths ← $(widths_cache)  $(size(total_widths))")
else
    total_widths = fill(NaN, want_shape)
    for (iter, r) in enumerate(REALIZATIONS)
        total_widths[iter, 1, :] = yr_2015_widths                # 2015 baseline (same for all r)
        f = "$(posteriors_directory)/R_$(r)_Posterior_Dict.jld2"
        isfile(f) || continue
        post = JLD2.load(f, "post_data")
        for (s, y) in enumerate(GAPPED_YEARS)
            haskey(post, y) && (total_widths[iter, s + 1, :] = ci_width(post[y]))
        end
    end
    if any(isnan, total_widths)
        @warn "total_widths incomplete: $(count(isnan, total_widths) ÷ 6) realization×year cells not finished yet (MCMC still running) — NOT caching. Re-run as more finish."
        println("Computed total_widths (incomplete)  $(size(total_widths))")
    else
        @save widths_cache total_widths
        println("Computed + cached total_widths → $(widths_cache)  $(size(total_widths))")
    end
end

In [ ]:
# Across-realization summary of the CI widths: per year-slot & parameter, the mean and a credible
# band. Currently running (NaN) realizations are skipped; a fully-unran slot stays NaN instead of erroring.
# Returns (mean_width[slot,param], band_up_low[slot,param,2]). Reusable for any total_widths array.
function summarize_widths(total_widths; band=(0.83, 0.17))
    nslot, npar = size(total_widths, 2), size(total_widths, 3)
    mean_width, band_up_low = zeros(nslot, npar), zeros(nslot, npar, 2)
    for i in 1:nslot, j in 1:npar
        col = filter(!isnan, total_widths[:, i, j])
        if isempty(col)
            mean_width[i, j] = NaN; band_up_low[i, j, :] .= NaN
        else
            mean_width[i, j] = mean(col); band_up_low[i, j, :] = quantile(col, collect(band))
        end
    end
    return mean_width, band_up_low
end

mean_width, un_cred_int_66_up_low = summarize_widths(total_widths)

In [ ]:
# FIGURE: parametric learning — 90%-CI width vs analysis year, per parameter (2×3): across-realization
# mean (green) + 66% band + example trajectories. Uses config TITLE_DICT/FONTSIZE and the cell-18
# mean_width / un_cred_int_66_up_low.
plot_trajectories = true                         # false → mean + band only (swaps filename + legend anchor too)
traj_suffix       = plot_trajectories ? "with" : "without"
output_file_name  = "../outputs/plots/Parametric_Learning_Plots/parametric_learning_$(traj_suffix)_trajectories$(TAG).pdf"
mkpath(dirname(output_file_name))

# figure-specific axis styling: per-parameter y-ticks; y-limit = 2015 baseline width × factor
tick_dict = Dict(1=>[6e3,1.2e4,1.8e4,2.4e4,3e4,3.6e4,4.2e4], 2=>[0.02,0.06,0.10,0.14,0.18,0.22],
                 3=>[0.05,0.10,0.15,0.20,0.25,0.30],         4=>[0.02,0.06,0.10,0.14,0.18,0.22,0.26],
                 5=>[0.5,1.0,1.5,2.0,2.5,3.0,3.5],           6=>[2,8,14,20,26,32,38])
factor    = plot_trajectories ? 1.325 : 1.2
ylim_dict = Dict(k => [0.0, yr_2015_widths[k] * factor] for k in 1:6)

selections        = [8,20,39,44,49,70,78,92] #    # example realizations drawn as individual trajectories
letters           = ["a","b","c","d","e","f"]

fig, ax = PyPlot.subplots(nrows=2, ncols=3, figsize=(19, 11.5), dpi=300,
    gridspec_kw=Dict("height_ratios"=>[1,1], "width_ratios"=>[1,1,1], "wspace"=>0.19, "hspace"=>0.25))

for i in 1:2, j in 1:3
    var_idx = tuple_to_idx(i, j); axij = ax[i, j]

    axij.plot(years_slots, mean_width[:, var_idx], color="green", lw=2.5,
              label="Across-trajectory mean", zorder=601)
    axij.fill_between(years_slots, un_cred_int_66_up_low[:, var_idx, 1], un_cred_int_66_up_low[:, var_idx, 2],
                      color="palegreen", alpha=0.3, label="66% credible interval", zorder=600)
    if plot_trajectories
        for (t, iter) in enumerate(selections)
            axij.plot(years_slots, total_widths[iter, :, var_idx], marker="o", markersize=4, ls="dashed",
                      color="black", markerfacecolor="green",
                      label = t == 1 ? "Individual trajectory" : "_nolegend_")
        end
    end

    axij.set_title(TITLE_DICT[var_idx], fontsize=1.2*FONTSIZE, pad=10)
    axij.ticklabel_format(style="sci", scilimits=(-2,4), useMathText=true)
    axij.tick_params(axis="both", which="major", labelsize=FONTSIZE*0.8)
    axij.xaxis.offsetText.set_fontsize(FONTSIZE*0.8); axij.yaxis.offsetText.set_fontsize(FONTSIZE*0.8)
    axij.set_yticks(tick_dict[var_idx]); axij.set_ylim(ylim_dict[var_idx])
    axij.set_xlim([2015, 2300]); axij.set_xticks([2050,2100,2150,2200,2250,2300])
    axij.annotate("2015", xy=(2015, 0), xycoords=("data","axes fraction"), xytext=(1991,-0.052), fontsize=FONTSIZE*0.8)
    axij.grid(true, alpha=0.5, zorder=1)
    axij.text(j == 1 ? -0.13 : -0.1, 1.095, "($(letters[var_idx]))", ha="left", va="top",
              transform=axij.transAxes, fontsize=FONTSIZE)
end

handles, labels = ax[1,1].get_legend_handles_labels()
keep = filter(i -> i ≤ length(handles), [1,2,3])
fig.legend([handles[i] for i in keep], [labels[i] for i in keep], loc="lower center", ncol=3,
           fontsize=0.8*FONTSIZE, bbox_to_anchor=(0.512, 0.92),
           bbox_transform=PyPlot.gcf().transFigure, frameon="True", framealpha=1)
fig.supxlabel("Analysis year", fontsize=FONTSIZE*1.2, y=0.036)
fig.supylabel("90% credible interval width", fontsize=FONTSIZE*1.2, x=0.066)

plt.savefig(output_file_name); println("Wrote $(output_file_name)")

################################################################################
################################################################################
################################################################################

# SLR PCA Projection

################################################################################
################################################################################
################################################################################

In [ ]:
# VAF PCA GP emulator: 5 per-PC GPs + a 285-step PCA basis (2016–2300), mapping calibrated
# parameters → SLR(mm) trajectories. Used to project each configuration's posteriors forward.
gps, PC_min, PC_max, pca, num_pcs = JLD2.load("../inputs/emulators/VAF_PCA_GP_Emulator.jld2",
    "gps", "PC_min", "PC_max", "p", "num_pcs")

# per-PC GP predictive mean & std (σ = √variance) at scaled inputs X (6×n)
predict_gp_eml(gp, X) = (mv = GaussianProcesses.predict_f(gp, X); (mv[1], sqrt.(mv[2])))
# truncated PCA reconstruction from the first J PC scores W → full 285-step trajectory
reconstruct_pcs(pca, W, J=length(W)) =
    MultivariateStats.decentralize(MultivariateStats.projection(pca)[:, 1:J] * W, mean(pca))

In [ ]:
# Emulator input scaling — vectors in PARAMS order. Ranges are identical to the calibration's
# X_mins/X_maxs (verified across both design CSVs), so we reuse config rather than refitting scalers.
param_mins = [X_mins[p] for p in PARAMS]
param_maxs = [X_maxs[p] for p in PARAMS]

In [ ]:
# Project one realization's calibrated posteriors forward to SLR(mm) trajectories via the VAF PCA GP
# emulator. Emulator (code) uncertainty is propagated ONLY for code-on configurations (A,B): each PC is
# drawn ~Normal(mean, σ). For code-off configurations (C,D) the GP posterior MEAN is used — no code
# uncertainty anywhere in the projection, matching FL_CODE_NOISE=0 in calibration + future-obs.
# Writes one file per calibration year → <path_to_projections>/R_<r>/<r>-year<yr>pred_VAF.jld2.
const PROJ_SAMPLE_SIZE = 5000     # posterior draws kept per year (systematic thinning → low autocorrelation)

function project_realization(r, post_dict)
    outdir = "$(path_to_projections)/R_$(r)"; mkpath(outdir)
    ntime  = size(MultivariateStats.projection(pca), 1)                # 285
    for (yr, chain) in post_dict
        step  = max(1, fld(size(chain, 1), PROJ_SAMPLE_SIZE))          # every-kth draw
        draws = chain[1:step:end, :]
        size(draws, 1) > PROJ_SAMPLE_SIZE && (draws = draws[1:PROJ_SAMPLE_SIZE, :])
        αs = (draws .- param_mins') ./ (param_maxs' .- param_mins')     # N×6, scaled to [0,1]

        Random.seed!(11)                                               # reproducible PC draws (code-on only)
        vaf = Matrix{Float64}(undef, ntime, size(draws, 1))            # 285 × N
        for s in 1:size(αs, 1)
            α  = reshape(αs[s, :], :, 1)                               # 6×1 GP input
            pc = [ let (μ, σ) = predict_gp_eml(gps[i], α)
                       m = only(μ) * (PC_max[i] - PC_min[i]) + PC_min[i]   # PC posterior mean (unscaled)
                       cfg.code ? rand(Normal(m, only(σ) * (PC_max[i] - PC_min[i]))) : m  # code-on: +σ ; code-off: mean only
                   end for i in 1:num_pcs ]
            vaf[:, s] = reconstruct_pcs(pca, pc, num_pcs)
        end
        sample_post_mm_ssp5 = vaf ./ (-362)                            # VAF(Gt) → SLR(mm), sign convention
        @save "$(outdir)/$(r)-year$(yr)pred_VAF.jld2" sample_post_mm_ssp5
    end
end

In [ ]:
# Generate SLR projections for the CURRENT configuration. Idempotent/resumable: skips realizations whose
# 19 year-files already exist; FORCE_PROJECT=true regenerates. Path carries TAG, so configurations don't collide.
# NOTE: expensive (100 × 19 × 5000 emulator draws) — run once per configuration, after its posteriors are complete.
FORCE_PROJECT = false
mkpath(path_to_projections)
is_complete(r) = (d = "$(path_to_projections)/R_$(r)";
                  isdir(d) && count(f -> endswith(f, "pred_VAF.jld2"), readdir(d)) == length(GAPPED_YEARS))
todo = FORCE_PROJECT ? collect(REALIZATIONS) : filter(r -> !is_complete(r), REALIZATIONS)
println("Projecting $(length(todo)) realizations ($(length(REALIZATIONS) - length(todo)) already complete) → $(path_to_projections)")
for r in todo
    post_dict = JLD2.load("$(path_to_posteriors)/R_$(r)_Posterior_Dict.jld2", "post_data")
    project_realization(r, post_dict)
    println("  done R$r ($(length(post_dict)) years)")
end

################################################################################
################################################################################
################################################################################

# SLR Learning Plots

################################################################################
################################################################################
################################################################################

In [ ]:
# SLR learning widths: for each prediction year, the 90%-CI width (mm) of the projected SLR at that
# year vs calibration (analysis) year, per trajectory. slr_widths[py] = [100 × n_cal_years], column 1 =
# 2015 baseline. Loading pred_VAF is expensive → cache once, but ONLY when complete: missing (still-
# ungenerated) projections stay NaN and the cache is skipped. FORCE_SLR_WIDTHS=true recomputes.
PRED_YEARS       = [2100, 2200, 2300]
FORCE_SLR_WIDTHS = false
slr_cache        = "$(path_to_projections)/slr_learning_widths.jld2"
cal_range(py)    = collect(range(2015, step=15, length=fld(py - 2015, 15) + 1))   # calibration years ≤ py
col(W, f)        = [ (v = filter(!isnan, W[:, k]); isempty(v) ? NaN : f(v)) for k in 1:size(W, 2) ]  # NaN-skipping column reduce

function build_slr_widths()
    present = JLD2.load("../inputs/params/2015_SLR_Projections_$(cfg.code ? "codeon" : "codeoff").jld2", "present2015_post_mm_ssp5")
    widths, present_width, complete = Dict{Int,Matrix{Float64}}(), Dict{Int,Float64}(), true
    for py in PRED_YEARS
        row = yrs_dict[py]
        _, l0, u0 = calculate_quantiles_mean(present, row); present_width[py] = u0 - l0
        years = cal_range(py)
        w = fill(NaN, length(REALIZATIONS), length(years))
        for (iter, r) in enumerate(REALIZATIONS)
            w[iter, 1] = present_width[py]                                        # 2015 baseline
            for (c, yr) in enumerate(years[2:end])
                f = "$(path_to_projections)/R_$(r)/$(r)-year$(yr)pred_VAF.jld2"
                 isfile(f) || (complete = false; continue)
                _, lo, hi = calculate_quantiles_mean(JLD2.load(f, "sample_post_mm_ssp5"), row)
                w[iter, c + 1] = hi - lo
            end
        end
        widths[py] = w
    end
    return widths, present_width, complete
end

if !FORCE_SLR_WIDTHS && isfile(slr_cache)
    slr_widths, present_width = JLD2.load(slr_cache, "slr_widths", "present_width")
    println("Loaded cached SLR learning widths ← $slr_cache")
else
    slr_widths, present_width, complete = build_slr_widths()
    if complete
        @save slr_cache slr_widths present_width
        println("Computed + cached SLR learning widths → $slr_cache  (years: $PRED_YEARS)")
    else
        @warn "SLR widths incomplete — projections not fully generated for this configuration; NOT caching. Run the projection driver, then re-run."
        println("Computed SLR learning widths (incomplete) — years: $PRED_YEARS")
    end
end

In [ ]:
# one trajectory's SLR-width curve (dashed black line, colored markers); labelled only for the first
function plot_slr_trajectory(ax, x, y, is_first, color)
    ax.plot(x, y, marker="o", markersize=6.6, color="black", ls="dashed", markerfacecolor=color,
            label = is_first ? "Individual trajectory" : "_nolegend_")
end

In [ ]:
# FIGURE: SLR learning — 90%-CI width (mm) of projected SLR vs calibration year, one panel per prediction
# year (2100/2200/2300): across-trajectory mean + 66% band + (optional) example trajectories.
plot_trajectories = true                          # false → mean + band only (swaps filename + legend anchor too)
traj_suffix       = plot_trajectories ? "with" : "without"
output_file_name  = "../outputs/plots/SLR_Learning_Plots/SLR_learning_$(traj_suffix)_trajectories$(TAG).pdf"
mkpath(dirname(output_file_name))
FONTSIZE_SLR  = 26
selections    = [13, 22, 40, 42, 58, 70, 78]      # example realizations drawn as trajectories
legend_anchor = plot_trajectories ? (0.785, 0.712) : (0.765, 0.78)

color_dict      = Dict(2100=>"darkblue", 2200=>"chocolate", 2300=>"purple")
fill_color_dict = Dict(2100=>"dodgerblue", 2200=>"orange", 2300=>"plum")
xticks_dict     = Dict(2100=>[2015,2030,2045,2060,2075,2090],
                       2200=>[2015,2045,2075,2105,2135,2165,2195],
                       2300=>[2015,2050,2100,2150,2200,2250,2300])
yticks_dict     = Dict(2100=>[1.5,3,4.5,6,7.5], 2200=>[12,24,36,48,60,72], 2300=>[14,28,42,56,70,84])
ylim_dict       = Dict(2100=>[0.0, present_width[2100]*1.35],
                       2200=>[0.0, present_width[2200]*1.2],
                       2300=>[0.0, present_width[2300]*1.3])
letters = ["a","b","c"]

fig, axes = PyPlot.subplots(nrows=1, ncols=3, figsize=(32, 6)); fig.subplots_adjust(wspace=0.14, hspace=0.0)

for (a, py) in enumerate(PRED_YEARS)
    ax = axes[a]; W = slr_widths[py]; years = cal_range(py)

    if plot_trajectories
        for iter in selections
            plot_slr_trajectory(ax, years, W[iter, :], iter == selections[1], color_dict[py])
        end
    end
    ax.plot(years, col(W, mean), color=color_dict[py], lw=2.5, label="Across-trajectory mean", zorder=500)
    ax.fill_between(years, col(W, x->quantile(x,0.83)), col(W, x->quantile(x,0.17)),
                    color=fill_color_dict[py], alpha=0.3, label="66% credible interval", zorder=499)

    ax.set_xlabel("Analysis year", fontsize=FONTSIZE_SLR*0.82, labelpad=10)
    a == 1 && ax.set_ylabel("90% credible interval width (mm)", fontsize=FONTSIZE_SLR*0.82, labelpad=12)
    ax.tick_params(axis="both", which="major", labelsize=FONTSIZE_SLR*0.68)
    ax.set_xlim(2015, years[end]); ax.set_ylim(ylim_dict[py])
    ax.set_xticks(xticks_dict[py]); ax.set_yticks(yticks_dict[py])
    ax.grid(true, alpha=0.5, zorder=1)
    ax.text(a == 1 ? -0.13 : -0.1, 1.08, "($(letters[a]))", ha="left", va="top",
            transform=ax.transAxes, fontsize=FONTSIZE_SLR)

    handles, labels = ax.get_legend_handles_labels()
    order = filter(i -> i ≤ length(handles), plot_trajectories ? [2,3,1] : [1,2])
    ax.legend([handles[i] for i in order], [labels[i] for i in order], loc="lower center", ncol=1,
              fontsize=FONTSIZE_SLR*0.675, bbox_to_anchor=legend_anchor, frameon="True", framealpha=1)
end

fig.savefig(output_file_name; bbox_inches="tight", dpi=300); println("Wrote $output_file_name")

################################################################################
################################################################################
################################################################################

# SLR Sensitivity Plots

################################################################################
################################################################################
################################################################################

In [ ]:
# SLR 90%-CI width at `pred_year` (mm) for each realization, using projections calibrated to `calib_year`.
# NaN for realizations whose projection isn't generated yet (mid-run safe).
function sensitivity_widths(pred_year, calib_year)
    row = yrs_dict[pred_year]
    w = fill(NaN, length(REALIZATIONS))
    for (i, r) in enumerate(REALIZATIONS)
        f = "$(path_to_projections)/R_$(r)/$(r)-year$(calib_year)pred_VAF.jld2"
        isfile(f) || continue
        _, lo, hi = calculate_quantiles_mean(JLD2.load(f, "sample_post_mm_ssp5"), row)
        w[i] = hi - lo
    end
    return w
end

In [ ]:
# PAPER FIGURE: SLR 90%-CI width at `pred_year` vs each true parameter (2×3), given calibration to
# `calib_year`; lowess trend per panel. X-axis ranges are parameter ranges (config-independent).
function plot_param_sensitivity(pred_year, calib_year; save=true)
    widths = sensitivity_widths(pred_year, calib_year)
    keep = .!isnan.(widths)
    any(keep) || (@warn "No projections for ($pred_year,$calib_year); skipping."; return)
    !all(keep) && @warn "$(count(!, keep))/$(length(REALIZATIONS)) realizations have no projection at calibration year $calib_year yet (MCMC still running) — plotting $(count(keep))."

    tick_dict = Dict(1=>[1.0e5,1.15e5,1.3e5,1.45e5], 2=>[0.1,0.15,0.2,0.25,0.3], 3=>[0.8,0.9,1,1.1],
                     4=>[0.84,0.90,0.96,1.02,1.08], 5=>[9.17,10,11,12], 6=>[12,20,30,40,50])
    xlim_dict = Dict(1=>[0.975e5,1.58e5], 2=>[0.09,0.328], 3=>[0.785,1.12],
                     4=>[0.815,1.11], 5=>[9.03,12.6], 6=>[9.5,58])
    letters = ["a","b","c","d","e","f"]

    fig, ax = PyPlot.subplots(nrows=2, ncols=3, figsize=(18, 11), dpi=300,
        gridspec_kw=Dict("height_ratios"=>[1,1], "width_ratios"=>[1,1,1], "wspace"=>0.175, "hspace"=>0.25))
    for j in 1:6
        rI, cI = (j-1) ÷ 3 + 1, (j-1) % 3 + 1; axi = ax[rI, cI]
        xvar = j == 5 ? log.(θ_mat[keep, j]) : θ_mat[keep, j]
        seaborn.regplot(x=xvar, y=widths[keep], lowess=true,
            scatter_kws=Dict("alpha"=>0.85, "s"=>42, "color"=>"green", "edgecolor"=>"gray",
                             "linewidths"=>0.5, "plotnonfinite"=>false, "zorder"=>2),
            line_kws=Dict("color"=>"red", "lw"=>1, "alpha"=>0.8), ax=axi)
        axi.set_title(string(TITLE_DICT[j]), fontsize=1.2*FONTSIZE, pad=10)
        axi.ticklabel_format(style="sci", scilimits=(-2,3), useMathText=true)
        axi.tick_params(axis="both", which="major", labelsize=FONTSIZE*0.8)
        axi.xaxis.offsetText.set_fontsize(FONTSIZE*0.8); axi.yaxis.offsetText.set_fontsize(FONTSIZE*0.8)
        axi.set_xticks(tick_dict[j]); axi.set_xlim(xlim_dict[j])
        axi.locator_params(tight=true, nbins=6); axi.grid(true, alpha=0.5, zorder=1)
        axi.text(cI == 1 ? -0.13 : -0.1, 1.095, "($(letters[j]))", ha="left", va="top",
                 transform=axi.transAxes, fontsize=FONTSIZE)
    end
    out = "../outputs/plots/Sensitivity_Plots/SLR_sensitivity_wrt_parameters_$(pred_year)_calib_yr_$(calib_year)$(TAG).pdf"
    save && (mkpath(dirname(out)); fig.savefig(out; bbox_inches="tight", dpi=300); println("Wrote $out"))
    PyPlot.close(fig); return out
end

In [ ]:
# # OPTIONAL (not in paper — kept for completeness): all 15 parameter pairs (3×5), points colored by the
# # SLR width at `pred_year` (calibration to `calib_year`). Color limits adapt to the width range.
# function plot_pairwise_sensitivity(pred_year, calib_year; save=true)
#     widths = sensitivity_widths(pred_year, calib_year); keep = .!isnan.(widths)
#     any(keep) || (@warn "No projections for ($pred_year,$calib_year); skipping."; return)
#     !all(keep) && @warn "$(count(!, keep))/$(length(REALIZATIONS)) realizations have no projection at calibration year $calib_year yet (MCMC still running) — plotting $(count(keep))."

#     W, X = widths[keep], θ_mat[keep, :]
#     FSZ = 27
#     tick_dict = Dict(1=>[1.0e5,1.15e5,1.3e5,1.45e5], 2=>[0.1,0.15,0.2,0.25,0.3], 3=>[0.8,0.9,1,1.1],
#                      4=>[0.84,0.90,0.96,1.02,1.08], 5=>[9.17,10,11,12], 6=>[12,20,30,40,50,58])
#     xlim_dict = Dict(1=>[1.02e5,1.485e5], 2=>[0.108,0.333], 3=>[0.78,1.148],
#                      4=>[0.81,1.095], 5=>[9.05,12.6], 6=>[10,60])
#     cmap = plt.get_cmap("viridis")
#     vmin, vmax = quantile(W, [0.05, 0.95])                 # robust; adapts per config
#     norm_ = mpl.colors.Normalize(vmin=vmin, vmax=vmax, clip=true)
#     pairs = [(a,b) for a in 1:6 for b in a+1:6]            # 15

#     fig, ax = PyPlot.subplots(nrows=3, ncols=5, figsize=(32, 18))
#     for (idx, (xk, yk)) in enumerate(pairs)
#         rI, cI = 1 + (idx-1) ÷ 5, 1 + (idx-1) % 5; axi = ax[rI, cI]
#         xv = xk == 5 ? log.(X[:, xk]) : X[:, xk]
#         yv = yk == 5 ? log.(X[:, yk]) : X[:, yk]
#         axi.set_rasterization_zorder(0)
#         axi.scatter(xv, yv; s=105, c=W, cmap=cmap, norm=norm_, edgecolor="gray", linewidth=0.5, rasterized=true)
#         axi.set_xlabel(string(TITLE_DICT[xk]), fontsize=FSZ, labelpad=10)
#         axi.set_ylabel(string(TITLE_DICT[yk]), fontsize=FSZ, labelpad=10)
#         axi.ticklabel_format(style="sci", scilimits=(-3,3), useMathText=true)
#         axi.tick_params(axis="both", which="major", labelsize=FSZ*0.8)
#         axi.xaxis.offsetText.set_fontsize(FSZ*0.8); axi.yaxis.offsetText.set_fontsize(FSZ*0.8)
#         axi.locator_params(tight=true, nbins=6); axi.grid(true, alpha=0.5, zorder=1)
#         axi.set_xticks(tick_dict[xk]); axi.set_yticks(tick_dict[yk])
#         axi.set_xlim(xlim_dict[xk]); axi.set_ylim(xlim_dict[yk])
#     end
#     plt.tight_layout(rect=[0.0, 0.08, 1.0, 1.0])
#     cax = fig.add_axes([0.10, 0.04, 0.80, 0.03])
#     sm = mpl.cm.ScalarMappable(norm=norm_, cmap=cmap); sm.set_array([])
#     cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
#     cbar.set_label("90% credible interval width", fontsize=FSZ*1.2, labelpad=10)
#     cbar.ax.tick_params(axis="x", which="major", labelsize=FSZ*0.8)
#     out = "../outputs/plots/Sensitivity_Plots/SLR_sensitivity_wrt_pairwise_parameters_$(pred_year)_calib_yr_$(calib_year)$(TAG).pdf"
#     save && (mkpath(dirname(out)); fig.savefig(out; bbox_inches="tight", dpi=300); println("Wrote $out"))
#     PyPlot.close(fig); return out
# end

In [ ]:
# The two individual-parameter sensitivity figures in the paper — always regenerated.
PAPER_SENSITIVITY = [(2200, 2075), (2300, 2150)]     # (prediction year, calibration year)
for (py, cy) in PAPER_SENSITIVITY
    plot_param_sensitivity(py, cy)
end

# # --- optional experiments (uncomment to explore) ---
# plot_param_sensitivity(2300, 2075)                 # other (prediction, calibration) combinations
# plot_pairwise_sensitivity(2200, 2075)              # pairwise version (15 pairs; not in the paper)

################################################################################
################################################################################
################################################################################

# Posterior Convergence (Selected Trajectories)

################################################################################
################################################################################
################################################################################

In [ ]:
# One selected trajectory's parameter posteriors: mean + 90% band vs analysis year (3×2), truth as a
# diamond. Skips analysis years not yet finished for this realization (mid-run safe).
function plot_param_means_of_trajectory(real, posterior_dict, axis, title_dict, tick_dict, ylim_dict, color, fill_color, FONTSIZE_PARAM)
    _v(post, j) = j == 5 ? log.(post[:, j]) : post[:, j]
    years  = Float64[2015]
    means  = Dict(j => [mean(_v(θ_posterior_2015, j))]          for j in 1:6)
    uppers = Dict(j => [quantile(_v(θ_posterior_2015, j), 0.95)] for j in 1:6)
    lowers = Dict(j => [quantile(_v(θ_posterior_2015, j), 0.05)] for j in 1:6)
    for yr in GAPPED_YEARS
        haskey(posterior_dict, yr) || continue
        push!(years, yr)
        for j in 1:6
            v = _v(posterior_dict[yr], j)
            push!(means[j], mean(v)); push!(uppers[j], quantile(v, 0.95)); push!(lowers[j], quantile(v, 0.05))
        end
    end
    letters = ["a","b","c","d","e","f"]
    for j in 1:6
        rI, cI = (j-1) ÷ 2 + 1, (j-1) % 2 + 1; ax = axis[rI, cI]
        ax.spines["top"].set_visible(false); ax.spines["right"].set_visible(false)
        ax.tick_params(which="both", top=false, right=false, labeltop=false, labelright=false)
        ax.plot(years, means[j], label="Parameter posterior distribution mean", zorder=20, color=color, lw=3)
        ax.fill_between(years, uppers[j], lowers[j], alpha=0.5, label="90% credible interval", color=fill_color)
        ax.ticklabel_format(style="sci", scilimits=(-2,4), useMathText=true)
        ax.tick_params(axis="both", which="major", labelsize=FONTSIZE_PARAM*0.8)
        ax.xaxis.offsetText.set_fontsize(FONTSIZE_PARAM*0.8); ax.yaxis.offsetText.set_fontsize(FONTSIZE_PARAM*0.8)
        rI == 3 && ax.set_xlabel("Analysis year", fontsize=FONTSIZE_PARAM, labelpad=10)
        ax.set_ylabel(title_dict[j], fontsize=FONTSIZE_PARAM, labelpad=10)
        ax.set_yticks(tick_dict[j]); ax.set_ylim(ylim_dict[j])
        ax.set_xlim(2015, 2305); ax.set_xticks(rI == 3 ? [2015,2050,2100,2150,2200,2250,2300] : Float64[])
        ax.grid(true, alpha=0.25, zorder=1)
        ax.text(cI == 1 ? -0.13 : -0.1, 1.095, "($(letters[j]))", ha="left", va="top", transform=ax.transAxes, fontsize=FONTSIZE_PARAM)
        y_true = j == 5 ? log(θ_mat[real, j]) : θ_mat[real, j]
        ax.scatter([2305], [y_true]; marker = real == 58 ? "*" : "D",
                                     s = real == 58 ? 280 : 64, 
                                     c=color, edgecolors="k", linewidths=0.8, zorder=21, clip_on=false)
    end
end

In [ ]:
# One selected trajectory's SLR projection at `pred_year`: mean + 90% band vs analysis year. Skips
# analysis years whose projection isn't generated yet (mid-run safe).
function plot_slr_means_of_trajectory(path_to_proj, axis, r, pred_year, color, fill_color, FONTSIZE_SLR)
    pred_idx = yrs_dict[pred_year]
    n_cal    = fld(pred_year - 2015, 15) + 1
    cal_years = collect(range(2015, step=15, length=n_cal))
    years  = Float64[2015]
    means  = [mean(slr_2015_baseline[pred_idx, :])]
    uppers = [quantile(slr_2015_baseline[pred_idx, :], 0.95)]
    lowers = [quantile(slr_2015_baseline[pred_idx, :], 0.05)]
    for yr in cal_years[2:end]
        f = "$(path_to_proj)/$(r)-year$(yr)pred_VAF.jld2"
        isfile(f) || continue
        proj = JLD2.load(f, "sample_post_mm_ssp5")
        push!(years, yr)
        push!(means, mean(proj[pred_idx, :])); push!(uppers, quantile(proj[pred_idx, :], 0.95)); push!(lowers, quantile(proj[pred_idx, :], 0.05))
    end
    axis.plot(years, means, label="Projection distribution mean", zorder=20, color=color, lw=3)
    axis.fill_between(years, uppers, lowers, alpha=0.5, label="90% credible interval", color=fill_color)
    axis.set_xlim(2015, cal_years[end])
    axis.set_xticks(pred_year == 2300 ? [2015,2050,2100,2150,2200,2250,2300] : cal_years[1:1+floor(Int, pred_idx/100):end])
    axis.tick_params(axis="both", which="major", labelsize=FONTSIZE_SLR*0.85)
    axis.grid(true, alpha=0.25, zorder=1)
    axis.set_xlabel("Analysis year", fontsize=FONTSIZE_SLR, labelpad=10)
    axis.set_ylabel("Sea-level contribution at $(pred_year) (mm)", fontsize=FONTSIZE_SLR, labelpad=10)
end

In [ ]:
# FIGURES (paper "Posterior Convergence"): for a few selected trajectories, parameter posteriors and SLR
# projections (mean + 90% band) vs analysis year, converging toward the true value (diamond).
selections = [14, 24, 58, 77]        # few trajectories — more gets too busy

title_dict = TITLE_DICT
tick_dict  = Dict(1=>[100000,120000,140000], 2=>[0.15,0.20,0.25,0.30], 3=>[0.8,0.9,1,1.1],
                  4=>[0.8,0.9,1], 5=>[9.17,10,11,12], 6=>[20,30,40,50])
ylim_dict  = Dict(1=>[0.98e5,1.5e5], 2=>[0.093,0.34], 3=>[0.79,1.14], 4=>[0.79,1.08], 5=>[9.05,12.2], 6=>[11.5,58.5])
color_dict      = Dict(1=>"red", 2=>"green", 3=>"purple", 4=>"blue")
fill_color_dict = Dict(1=>"lightcoral", 2=>"palegreen", 3=>"plum", 4=>"lightblue")

gray_patch       = matplotlib.patches.Patch(color="gray", label="90% credible interval", alpha=0.5)
black_line_param = matplotlib.lines.Line2D([], [], lw=3, color="black", label="Parameter posterior distribution mean")
black_line_slr   = matplotlib.lines.Line2D([], [], lw=3, color="black", label="Projection distribution mean")

# --- parameter figure (3×2) ---
fig_params, ax_params = PyPlot.subplots(nrows=3, ncols=2, figsize=(28, 15), dpi=300,
    gridspec_kw=Dict("height_ratios"=>[1,1,1], "width_ratios"=>[1,1], "hspace"=>0.20, "wspace"=>0.15))
FONTSIZE_PARAM = 26
for (i, r) in enumerate(selections)
    posterior_dict = JLD2.load("$(path_to_posteriors)/R_$(r)_Posterior_Dict.jld2", "post_data")
    plot_param_means_of_trajectory(r, posterior_dict, ax_params, title_dict, tick_dict, ylim_dict,
                                   color_dict[i], fill_color_dict[i], FONTSIZE_PARAM)
end
ax_params[end].set_xlabel("Analysis year", fontsize=FONTSIZE_PARAM, labelpad=10)
fig_params.legend(handles=[black_line_param, gray_patch], loc="upper center", ncol=6, fontsize=0.9*FONTSIZE_PARAM,
                  bbox_to_anchor=(0.5, 0.962), bbox_transform=PyPlot.gcf().transFigure, frameon="True", framealpha=1)
param_out = "../outputs/plots/Posterior_Convergence_Plots/param_posterior_against_analysis_year$(TAG).pdf"
mkpath(dirname(param_out)); fig_params.savefig(param_out; dpi=300); plt.close(); println("Wrote $param_out")

# --- SLR figure (1×3 prediction years) ---
fig_slr, axes = PyPlot.subplots(nrows=1, ncols=3, figsize=(33, 6)); fig_slr.subplots_adjust(wspace=0.2, hspace=0.0)
FONTSIZE_SLR = 22
letters = ["a","b","c"]
for (a, chosen_year) in enumerate([2100, 2200, 2300])
    ax_slr = axes[a]
    for (i, r) in enumerate(selections)
        plot_slr_means_of_trajectory("$(path_to_slr_projections)/R_$(r)", ax_slr, r, chosen_year,
                                     color_dict[i], fill_color_dict[i], FONTSIZE_SLR)
    end
    a == 1 && ax_slr.set_ylim([-3.97, 4.75])
    xoff = a == 3 ? -0.157 : (a == 2 ? -0.145 : -0.149)
    ax_slr.text(xoff, 1.09, "($(letters[a]))", ha="left", va="top", transform=ax_slr.transAxes, fontsize=FONTSIZE_SLR*0.92)
    ax_slr.legend(handles=[black_line_slr, gray_patch], loc="upper center", ncol=1, fontsize=FONTSIZE_SLR*0.72,
                  bbox_to_anchor=(0.754, 1.013), frameon="True", framealpha=1)
end
slr_out = "../outputs/plots/Posterior_Convergence_Plots/projected_SLR_against_analysis_year$(TAG).pdf"
mkpath(dirname(slr_out)); fig_slr.savefig(slr_out; bbox_inches="tight", dpi=300); plt.close(); println("Wrote $slr_out")

## Emulator sensitivity diagnostic (J = ∂μ/∂θ, ΔJ)

Central-difference Jacobian of the per-year GP emulator's mean grounded-volume prediction w.r.t. each of the six parameters, evaluated at the present-day (2015) posterior mean, vs analysis year. `J` is the sensitivity; `ΔJ` (adjacent-analysis year increment) is the change between successive analysis years **roughness** that feeds the Wiener Fisher information. Pointwise emulator only; the smoothed (Savitzky–Golay) trend highlights the jitter. Mean is identical for `predict_f`/`predict_y`, so this figure is independent of the code-noise convention.

In [ ]:
# Emulator sensitivity J = dmu_vol/dtheta (central diff at the present-day posterior-mean theta), per
# analysis year, and its adjacent-year increment dJ (the per-year GP roughness driving the Wiener Fisher
# information). Pointwise-only + a Savitzky-Golay smoothed trend so the jitter reads against it.
sens_years = GAPPED_YEARS                                              # 2030,2045,...,2300
sens_ncol  = length(sens_years)
Xmn_s = [X_mins[p] for p in PARAMS]; Xmx_s = [X_maxs[p] for p in PARAMS]
theta_mean_scaled = (vec(mean(θ_posterior_2015, dims=1)) .- Xmn_s) ./ (Xmx_s .- Xmn_s)

gpd_sens = GPUtils.load_emulators("../inputs/emulators/cumulative_grounded_volume_change_gap_15_emulators.jld2")
gv_mins, gv_maxs = JLD2.load("../inputs/data/grd_vol_change_mins_maxs.jld2", "y_grd_vol_mins", "y_grd_vol_maxs")
Ymin_s = [gv_mins[Symbol("y_grd_vol_$y")] for y in sens_years]
Ymax_s = [gv_maxs[Symbol("y_grd_vol_$y")] for y in sens_years]
gpy_sens = [gpd_sens[y] for y in sens_years]
mu_vol(θ) = (r = reshape(collect(θ), length(PARAMS), 1);
             [GaussianProcesses.predict_f(gpy_sens[k], r)[1][1] for k in 1:sens_ncol] .* (Ymax_s .- Ymin_s) .+ Ymin_s)

function sens_jacobian(μf, θ; h=1e-3)                                  # central difference J[year, param]
    J = zeros(sens_ncol, length(PARAMS))
    for k in 1:length(PARAMS)
        θp = copy(collect(θ)); θm = copy(collect(θ)); θp[k] += h; θm[k] -= h
        J[:, k] = (μf(θp) .- μf(θm)) ./ (2h)
    end
    J
end
J_sens  = sens_jacobian(mu_vol, theta_mean_scaled)
dJ_sens = mapslices(diff, J_sens, dims=1)
sgolay  = pyimport("scipy.signal").savgol_filter
smooth_cols(M) = hcat([sgolay(M[:, k], min(7, isodd(size(M,1)) ? size(M,1) : size(M,1)-1), 2) for k in 1:size(M,2)]...)

println("Emulator roughness  R_k = Σ(ΔJ)²  (higher ⇒ jitterier sensitivity):")
for k in 1:length(PARAMS); println("  ", rpad(string(PARAMS[k]), 12), round(sum(dJ_sens[:, k].^2), sigdigits=4)); end

function sensitivity_figure(raw, sm, xv, ylab, fname)
    letters = ["a","b","c","d","e","f"]
    fig, axs = PyPlot.subplots(nrows=2, ncols=3, figsize=(19,11.5), dpi=300,
        gridspec_kw=Dict("wspace"=>0.19, "hspace"=>0.25))

    for k in 1:length(PARAMS)
        i, j = (k-1)÷3 + 1, (k-1)%3 + 1
        ax = axs[i,j]

        ax.axvspan(2180, 2225, color="0.85", alpha=0.6, zorder=0)
        ax.axhline(0, color="0.6", lw=0.8, zorder=1)
        ax.plot(xv, raw[:,k], color="#1f5fa8", lw=2.2, marker="o", ms=6,
                label="Pointwise sensitivity", zorder=3)
        ax.plot(xv, sm[:,k], color="#c0392b", lw=2.4, ls="--",
                label="Smoothed trend", zorder=2)

        ax.set_title(TITLE_DICT[k], fontsize=1.2*FONTSIZE, pad=10)
        ax.ticklabel_format(axis="y", style="sci", scilimits=(-2,3), useMathText=true)
        ax.tick_params(axis="both", which="major", labelsize=0.8*FONTSIZE)
        ax.xaxis.offsetText.set_fontsize(0.8*FONTSIZE)
        ax.yaxis.offsetText.set_fontsize(0.8*FONTSIZE)

        ax.set_xlim(2030,2300); ax.set_xticks(2050:50:2300)
        ax.grid(true, alpha=0.5, zorder=0)
        ax.text(j == 1 ? -0.13 : -0.10, 1.095, "($(letters[k]))",
                ha="left", va="top", transform=ax.transAxes, fontsize=FONTSIZE)
    end

    handles, labels = axs[1,1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=0.8*FONTSIZE,
               bbox_to_anchor=(0.512,0.92), bbox_transform=fig.transFigure,
               frameon=true, framealpha=1)

    fig.supxlabel("Analysis year", fontsize=1.2*FONTSIZE, y=0.036)
    fig.supylabel(ylab, fontsize=1.2*FONTSIZE, x=0.060)

    d = "../outputs/plots/sensitivity_diagnostics"; mkpath(d)
    fig.savefig("$d/$fname.pdf", bbox_inches="tight")
    println("Wrote $d/$fname.pdf")
    fig
end

# sensitivity_figure(J_sens,  smooth_cols(J_sens),  sens_years, "Emulator sensitivity", "sensitivity_J")
# sensitivity_figure(dJ_sens, smooth_cols(dJ_sens), sens_years[2:end], "Change between analysis years", "sensitivity_dJ");

In [ ]:
# FIGURE: combined emulator sensitivity — J (blue, left) + change between successive analysis years change ΔJ (red, right), per parameter (2×3).
# Square panels with aligned left/right y-ticks; shared colored y-labels replace the legend.

FONTSIZE_Sensitivity = 23

_nice(x) = (x <= 0 ? 1.0 : begin
    e = floor(log10(x)); b = x/10.0^e
    m = b <= 1 ? 1.0 : b <= 2 ? 2.0 : b <= 2.5 ? 2.5 : b <= 5 ? 5.0 : 10.0
    m*10.0^e
end)

function align_right!(axL, axR, dvals; nticks=5)
    loL, hiL = axL.get_ylim()
    tL = [t for t in axL.get_yticks() if loL-1e-9 <= t <= hiL+1e-9]
    length(tL) < 2 && return

    fL = [(t-loL)/(hiL-loL) for t in tL]
    dmin, dmax = extrema(dvals)
    dmax-dmin < 1e-12 && (dmax = dmin + 1)

    step = _nice((dmax-dmin)/(nticks-1))
    start = floor(dmin/step)*step
    while start + (nticks-1)*step < dmax-1e-9
        step = _nice(2step); start = floor(dmin/step)*step
    end

    tR = [start+(i-1)*step for i in 1:nticks]
    span = (tR[end]-tR[1])/(fL[end]-fL[1])
    loR = tR[1]-fL[1]*span
    axR.set_ylim(loR, loR+span); axR.set_yticks(tR)
end

let
    letters = ["a","b","c","d","e","f"]
    blue, red = "#1f5fa8", "#c0392b"

    fig, axs = PyPlot.subplots(nrows=2, ncols=3, figsize=(21,12), dpi=300,
        gridspec_kw=Dict("height_ratios"=>[1,1], "width_ratios"=>[1,1,1], "wspace"=>0.32, "hspace"=>0.3))

    for k in 1:length(PARAMS)
        i, j = (k-1)÷3 + 1, (k-1)%3 + 1
        ax = axs[i,j]

        ax.axvspan(2180,2225, color="0.85", alpha=0.6, zorder=0)
        ax.axhline(0, color="0.6", lw=0.8, zorder=1)

        ax.plot(sens_years, J_sens[:,k], color=blue, lw=2.2, marker="o", ms=5, zorder=3)
        ax.set_title(TITLE_DICT[k], fontsize=1.2*FONTSIZE_Sensitivity, pad=9)
        ax.ticklabel_format(axis="y", style="sci", scilimits=(-2,3), useMathText=true)
        ax.tick_params(axis="both", which="major", labelsize=0.8*FONTSIZE_Sensitivity)
        ax.tick_params(axis="y", labelcolor=blue, colors=blue)
        ax.xaxis.offsetText.set_fontsize(0.8*FONTSIZE_Sensitivity)
        ax.yaxis.offsetText.set_fontsize(0.8*FONTSIZE_Sensitivity)
        ax.yaxis.offsetText.set_color(blue)
        ax.spines["left"].set_color(blue)
        ax.set_xlim(2030,2300); ax.set_xticks(2050:50:2300)
        ax.grid(true, alpha=0.45, zorder=0)

        ax.text(j == 1 ? -0.12 : -0.09, 1.08, "($(letters[k]))",
                ha="left", va="top", transform=ax.transAxes, fontsize=FONTSIZE_Sensitivity)

        axr = ax.twinx()
        axr.plot(sens_years[2:end], dJ_sens[:,k], color=red, lw=1.8, ls="--",
                 marker="s", ms=4.5, alpha=0.9, zorder=2)
        align_right!(ax, axr, dJ_sens[:,k]; nticks=5)
        axr.ticklabel_format(axis="y", style="sci", scilimits=(-2,3), useMathText=true)
        axr.tick_params(axis="y", which="major", labelsize=0.75*FONTSIZE_Sensitivity,
                        labelcolor=red, colors=red)
        axr.yaxis.offsetText.set_fontsize(0.75*FONTSIZE_Sensitivity)
        axr.yaxis.offsetText.set_color(red)
        axr.spines["right"].set_color(red)
    end

    fig.supxlabel("Analysis year", fontsize=1.2*FONTSIZE_Sensitivity, y=0.02)
    fig.supylabel("Emulator sensitivity", fontsize=1.2*FONTSIZE_Sensitivity, x=0.072, color=blue)
    fig.text(0.935, 0.5, "Change between analysis years", rotation=270,
             ha="center", va="center", fontsize=1.05*FONTSIZE_Sensitivity, color=red)

    d = "../outputs/plots/sensitivity_diagnostics"; mkpath(d)
    fig.savefig("$d/sensitivity_J_and_dJ_combined.pdf")
    println("Wrote $d/sensitivity_J_and_dJ_combined.pdf")
end